In [1]:
import pandas as pd
import os
from dqn.dqn_config import HYPER_PARAMS
from colorama import Fore

## Remove redundant transition in Agent data and save it in separate folder

In [2]:
def clean_data(data_dir_name:str):
    ## Set data paths and create directory for cleaned data
    input_dir_path = os.path.join(HYPER_PARAMS.agent_data_dir,data_dir_name)
    output_dir_path = os.path.join(HYPER_PARAMS.agent_data_dir,data_dir_name+"_cleaned")
    os.makedirs(output_dir_path, exist_ok=True)

    ## Set max size of CSV file to 95 MiB
    max_file_size_bytes = 95 * 1024 * 1024

    ## Loading CSV file
    print(Fore.LIGHTCYAN_EX,"Loading CSV files...", Fore.RESET)
    # Get path of all CSV file 
    csv_files=[os.path.join(input_dir_path, f) for f in os.listdir(input_dir_path)  if f.endswith('.csv')]
    # Read and combine all CSV files into one DataFrame
    df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    print(f"Total rows before cleaning: {len(df)}")

    # # Remove redundant transitions
    print(Fore.LIGHTCYAN_EX, "Removing redundant transitions...", Fore.RESET)
    subset_cols = df.columns.drop("trans_idx")
    df = df.drop_duplicates(subset=subset_cols)
    print(f"Total rows after cleaning: {len(df)}")


    # # Saving cleaned data into multiple CSVs with max size constraint
    print(Fore.LIGHTCYAN_EX, "Saving cleaned data...", Fore.RESET)
    current_chunk = []
    current_size = 0
    file_index = 0

    header_size = len(df.iloc[:0].to_csv(index=False))

    for _, row in df.iterrows():
        row_csv = row.to_frame().T.to_csv(index=False, header=False)
        row_size = len(row_csv.encode("utf-8"))

        if current_size + row_size > max_file_size_bytes:
            out_path = os.path.join(output_dir_path, f"cleaned_{file_index}.csv")
            pd.DataFrame(current_chunk).to_csv(out_path, index=False)
            print(f"Saved {out_path}")
            file_index += 1
            current_chunk = []
            current_size = header_size

        current_chunk.append(row)
        current_size += row_size

    # Save remaining rows
    if current_chunk:
        out_path = os.path.join(output_dir_path, f"cleaned_{file_index}.csv")
        pd.DataFrame(current_chunk).to_csv(out_path, index=False)
        print(f"Saved {out_path}")

    print(Fore.YELLOW,"Done", Fore.RESET)


In [ ]:
clean_data(data_dir_name="transition_csv_DoubleDQN_OnRL_16e6")

### Check and free DataFrame RAM usage

In [4]:
def get_ram_usage(df):
    mb = df.memory_usage(deep=True).sum() / 1024**2
    gb = df.memory_usage(deep=True).sum() / 1024**3
    print(f"{mb:.2f} MB ({gb:.2f} GB)")

In [ ]:
input_dir_path = os.path.join(HYPER_PARAMS.agent_data_dir,"transition_csv_DoubleDQN_OnRL_16e6")
csv_files=[os.path.join(input_dir_path, f) for f in os.listdir(input_dir_path)  if f.endswith('.csv')]
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

get_ram_usage(df)

df.drop(df.index, inplace=True) 
get_ram_usage(df)

4546.02 MB (4.44 GB)
0.00 MB (0.00 GB)


## Generate seed for evaluation 

In [22]:
import random
from sumo_sim.sim_config import SimulationConfig as SC

random.seed(42)  # for reproducibility

seed_test = set()
while len(seed_test) < 50:
    s = str(random.randint(0, 30000))
    if s not in SC.seed_train:
        seed_test.add(s)

seed_test = list(seed_test)
print(seed_test)

['11149', '20952', '8024', '25018', '5231', '2848', '24132', '3648', '29234', '7055', '6515', '17856', '7164', '16559', '19309', '7623', '3358', '7223', '13848', '13825', '11029', '21295', '18390', '9115', '976', '1041', '24270', '9105', '23462', '22981', '819', '3070', '869', '19726', '17870', '4572', '24299', '9012', '24864', '13746', '28485', '212', '26523', '14719', '26405', '7314', '5094', '22876', '19349', '22174']
